<a href="https://colab.research.google.com/github/Danishh-ux/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Danishh-ux/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one content item (page), on one day, for one client** —
`report_date + client_hash_id + content_hash_id` in `fact_content_daily_performance`. For my
lane (Refresh / Content Opportunity Scoring), I aggregate that daily grain up to
**one row per content item** across a chosen window, joined to `dim_content` for static
content properties.

**Time window: `month=2026-03` (a mid-panel month), split into two internal halves:**
- **Decision-moment window:** March 1-15, 2026 -- everything my features come from.
- **Outcome window:** March 16-31, 2026 -- where my label comes from.

I deliberately keep both halves inside the *same* month partition rather than comparing two
different months, per the assignment's warning: the `_sample` table is the panel's sealed
final month, and I don't want to build label logic against it. Working within one mid-panel
month keeps my label-vs-feature split honest without touching the sealed test month at all.

I verify all of this with real queries below, not just assert it.

In [1]:
# Pinning down the exact window boundaries used everywhere below --
# no network needed for this, just making the contract concrete.
import pandas as pd

MONTH_PARTITION = "2026-03"
DECISION_START = pd.Timestamp("2026-03-01")
DECISION_END   = pd.Timestamp("2026-03-15")   # feature window: Mar 1-15
OUTCOME_START  = pd.Timestamp("2026-03-16")   # label window:   Mar 16-31
OUTCOME_END    = pd.Timestamp("2026-03-31")

print(f"Feature window: {DECISION_START.date()} .. {DECISION_END.date()} "
      f"({(DECISION_END - DECISION_START).days + 1} days)")
print(f"Label window:   {OUTCOME_START.date()} .. {OUTCOME_END.date()} "
      f"({(OUTCOME_END - OUTCOME_START).days + 1} days)")
print(f"Both windows sit inside partition month={MONTH_PARTITION} -- a mid-panel month,")
print("never the sealed final-month _sample table.")


Feature window: 2026-03-01 .. 2026-03-15 (15 days)
Label window:   2026-03-16 .. 2026-03-31 (16 days)
Both windows sit inside partition month=2026-03 -- a mid-panel month,
never the sealed final-month _sample table.


## 2. Fields: feature / label / context / excluded

**Tables I use:** `fact_content_daily_performance` (partition `month=2026-03`),
`dim_content`, `dim_clients`.

**What I'd predict (label/proxy):** `is_declining` -- 1 if a page's impressions in the
outcome window (Mar 16-31) fell to less than 80% of its decision-moment window (Mar 1-15),
else 0. This mirrors the starter pipeline's `is_declining_label` idea but built fresh from
warehouse daily data on a window I control, so I know exactly where every number comes from.

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (Mar 1-15 only) | **Feature** | Observed, known before the decision moment (Mar 16) |
| `word_count`, `content_type` (from `dim_content`) | **Feature** | Static content property, known at any time, not window-dependent |
| `content_created_at` (used to derive age-at-decision) | **Feature** (via derived age) | Known before the decision moment |
| `gsc_impressions` (Mar 16-31 only) | **Label source** | This is what `is_declining` is computed from -- never a feature (this is exactly the trap in section 3) |
| `content_hash_id`, `client_hash_id` | **Context** | Joins, grouping, client-holdout splits -- never fed to a model as a feature |
| `ga4_data_available` | **Context / filter** | Used to decide whether GA4-derived rows are usable, not as a feature itself |
| `health_score`, `priority_score`, `action_type`, any FlyRank product flag | **Excluded** | Not shipped in this release at all (confirmed in `docs/ml-intern-dataset-and-lane-guide.md` section 4) -- nothing to exclude by hand, but naming it because the trap in section 3 would recreate the same problem with `imp_last15` if I weren't careful |
| `ga4_sessions`, engagement columns | **Excluded (for now)** | This mid-panel month is early in some clients' `ga4_data_start` -- mixing them in without the `IS TRUE` filter would silently treat "not tracked yet" as "zero engagement" |

In [2]:
# The feature/label/context/excluded table above is the contract itself --
# this cell just re-states it as data so I can sanity-check it programmatically
# (real verification queries against the warehouse are in section 3 below).
contract = {
    "gsc_impressions/clicks/position (Mar 1-15)": "feature",
    "word_count / content_type (dim_content)": "feature",
    "content_created_at -> age at decision": "feature",
    "gsc_impressions (Mar 16-31)": "label_source",
    "content_hash_id / client_hash_id": "context",
    "ga4_data_available": "context_filter",
    "health_score / priority_score / action_type": "excluded_not_shipped",
    "ga4_sessions / engagement cols (this month)": "excluded_for_now",
}
import pandas as pd
pd.Series(contract, name="bucket").to_frame()


,bucket
gsc_impressions/clicks/position (Mar 1-15),feature
word_count / content_type (dim_content),feature
content_created_at -> age at decision,feature
gsc_impressions (Mar 16-31),label_source
content_hash_id / client_hash_id,context
ga4_data_available,context_filter
health_score / priority_score / action_type,excluded_not_shipped
ga4_sessions / engagement cols (this month),excluded_for_now


## 3. Verify it with queries (grain, counts, missing values, windows) + five features + the trap

Everything below runs against the real warehouse release on Hugging Face
(`hf://datasets/FlyRank/internship-warehouse`), partition `month=2026-03`. This needs your own
`HF_TOKEN` (Colab Secret, never pasted in a cell) -- see `SETUP.md` Moment 2 if you haven't set
it up. Structure:
- **3a** -- connect, then three verification queries: grain, row count + date span, availability
  (`IS TRUE`).
- **3b** -- the five-feature frame, one "available when?" line each.
- **3c** -- the trap: add the label's own source column as a "fake feature," watch the score
  jump toward perfect, then remove it and keep the honest number.

In [3]:
%pip -q install duckdb

import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    # single mid-panel month partition -- NOT the sealed final-month _sample table
    "fact_march":  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:14} {n:>12,} rows")


Paste your Hugging Face READ token (hf_...): ··········
dim_clients             104 rows
dim_content         519,606 rows
fact_march        9,841,378 rows


### 3a. Three verification queries

In [4]:
# Query 1 -- GRAIN: one row really is report_date x client x content (zero rows back = grain holds)
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print("Grain check (should be EMPTY):")
print(grain_check)

# Query 2 -- ROW COUNT + DATE SPAN for this slice
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(DISTINCT content_hash_id) AS n_content
    FROM {TABLES['fact_march']}
""").df()
print("\nRow count + date span:")
print(span)

# Query 3 -- AVAILABILITY, filtered with IS TRUE (never a plain '= TRUE' or 'NOT ...' --
# the flag can also be NULL, and IS TRUE / IS NOT TRUE is the only safe filter)
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_unavailable_or_null_rows
    FROM {TABLES['fact_march']}
""").df()
print("\nAvailability (IS TRUE):")
print(availability)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check (should be EMPTY):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []

Row count + date span:
    n_rows   min_date   max_date  n_clients  n_content
0  9841378 2026-03-01 2026-03-31         55     331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Availability (IS TRUE):
   total_rows  ga4_available_rows  ga4_unavailable_or_null_rows
0     9841378            413966.0                     9427412.0


### 3b. Five features, max -- one "available when?" line each

Built from the **decision-moment window only (Mar 1-15)** plus static content properties --
nothing from Mar 16-31, since that's where the label comes from.

In [6]:
features = con.sql(f"""
    WITH windowed AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date <  DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS imp_prev15,
            SUM(CASE WHEN f.report_date >= DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS imp_last15,
            SUM(CASE WHEN f.report_date <  DATE '2026-03-16' THEN f.gsc_clicks ELSE 0 END)      AS clk_prev15,
            AVG(CASE WHEN f.report_date <  DATE '2026-03-16' THEN f.gsc_avg_position END)       AS pos_prev15
        FROM {TABLES['fact_march']} f
        WHERE f.ga4_data_available IS TRUE OR f.ga4_data_available IS NULL
        GROUP BY 1, 2
    )
    SELECT w.*, c.word_count, c.content_type,
           DATE_DIFF('day', c.content_created_date, DATE '2026-03-16') AS content_age_days_at_decision
    FROM windowed w
    JOIN {TABLES['dim_content']} c ON c.content_hash_id = w.content_hash_id
    WHERE w.imp_prev15 >= 5   -- minimum volume floor, keeps pure noise out
""").df()

print(f"{len(features):,} content items with enough Mar 1-15 history")
features.head()

# Feature notes -- "available at the decision moment (Mar 16) because...":
# 1. imp_prev15   -- already observed search impressions from BEFORE Mar 16
# 2. clk_prev15   -- already observed clicks from BEFORE Mar 16
# 3. pos_prev15   -- already observed average position from BEFORE Mar 16
# 4. word_count   -- a static content property, true regardless of which day I ask
# 5. content_age_days_at_decision -- derived only from content_created_date (in the past)
#    and the fixed decision date Mar 16; needs no future information at all


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

102,639 content items with enough Mar 1-15 history


,client_hash_id,content_hash_id,imp_prev15,imp_last15,clk_prev15,pos_prev15,word_count,content_type,content_age_days_at_decision
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,111.0,70.0,0.0,5.222776,2999,keyword article,32
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,38.0,8.0,1.0,4.638889,3057,keyword article,32
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,219.0,680.0,1.0,3.737399,2855,keyword article,32
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,20.0,14.0,0.0,3.597222,3281,keyword article,32
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,1494.0,1614.0,0.0,6.156643,2779,keyword article,32


### 3c. The trap -- add a label-derived column on purpose

`is_declining` is defined FROM `imp_last15` (the Mar 16-31 outcome). Adding `imp_last15`
itself as a "feature" means the model just re-derives its own label -- watch the score jump
toward perfect, then delete it and keep only the honest number from the five clean features.

In [7]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

features["is_declining"] = (features["imp_last15"] < 0.8 * features["imp_prev15"]).astype(int)
print(features["is_declining"].value_counts())

clean_feats = ["imp_prev15", "clk_prev15", "pos_prev15", "word_count", "content_age_days_at_decision"]
X = features[clean_feats].fillna(0)
y = features["is_declining"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
honest_auc = roc_auc_score(yte, honest_model.predict_proba(Xte)[:, 1])
print(f"\nHonest AUC (5 clean features, no leakage): {honest_auc:.3f}")

# --- THE TRAP: sneak the label's own source column in as a "feature" ---
leaky_feats = clean_feats + ["imp_last15"]
Xl = features[leaky_feats].fillna(0)
Xltr, Xlte, yltr, ylte = train_test_split(Xl, y, test_size=0.3, random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000).fit(Xltr, yltr)
leaky_auc = roc_auc_score(ylte, leaky_model.predict_proba(Xlte)[:, 1])
print(f"Leaky AUC (imp_last15 included -- THE TRAP): {leaky_auc:.3f}")

print("\nThe leaky score jumps toward 1.0 because imp_last15 IS what the label was computed")
print("from -- the model isn't learning a pattern, it's just reading its own answer key back.")
print("Deleting imp_last15 and keeping the honest 5-feature AUC above is the real number I")
print("carry forward into Week 4/5.")


is_declining
1    56899
0    45740
Name: count, dtype: int64

Honest AUC (5 clean features, no leakage): 0.639
Leaky AUC (imp_last15 included -- THE TRAP): 1.000

The leaky score jumps toward 1.0 because imp_last15 IS what the label was computed
from -- the model isn't learning a pattern, it's just reading its own answer key back.
Deleting imp_last15 and keeping the honest 5-feature AUC above is the real number I
carry forward into Week 4/5.


## 4. Data limits

What this slice can never tell me:
- **A single 15-vs-15-day split inside one month is short.** A page can look "declining" here
  purely from a mid-month blip (a algorithm-update ripple, a seasonal dip, a competitor's
  temporary push) that a longer window would reveal as noise -- this is exactly the
  noise-vs-real-decline distinction the lane guide warns about (section 7).
- **Unbalanced panel.** Not every client has GSC history reaching back to March 2026 in the
  same way, and `ga4_data_start` varies per client -- clients who joined tracking late can
  look artificially "flat" here rather than genuinely stable, purely because they have less
  history behind them.
- **This month alone can't say anything about seasonality or persistence.** A decline that
  shows up in March could reverse in April; I can't tell from one month whether a drop
  persists, without checking additional months later in the project.
- **No causal claim available.** Nothing here proves *why* a page's impressions moved --
  only that they did. I can rank pages by this evidence, not diagnose the cause.

In [8]:
# Supporting check for the "unbalanced panel" limitation
coverage = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
    LIMIT 10
""").df()
print(coverage)
print("\nClients with a later gsc_data_start/ga4_data_start than March 2026 would contribute")
print("little or nothing to this slice -- worth checking before trusting a 'flat' verdict for them.")


            client_hash_id gsc_data_start ga4_data_start
0  client_9958f0a7ae1df715     2025-01-27     2025-10-29
1  client_ff644d8251367cbb     2025-01-27     2025-10-29
2  client_73cda7b4e4f265ea     2025-02-11     2026-03-24
3  client_fef1a8f436438636     2025-03-11     2026-03-06
4  client_62f4a7e64f5e0096     2025-06-07            NaT
5  client_b10cb2997d0c7c86     2025-06-18     2025-11-15
6  client_65de48885f4ef01b     2025-06-21     2026-02-19
7  client_c182d11e4862a37d     2025-06-21     2026-02-20
8  client_3197e6291363b4db     2025-06-29     2025-11-09
9  client_625b6439094e23e4     2025-07-01     2026-02-19

Clients with a later gsc_data_start/ga4_data_start than March 2026 would contribute
little or nothing to this slice -- worth checking before trusting a 'flat' verdict for them.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.